# PubChem → Gaussian Input File Pipeline

This notebook automates the generation of **Gaussian input files** (`.com`) and **SLURM submission scripts** (`.sh`) from a list of molecule names. It handles the entire workflow:

1. **Resolve** molecule names → PubChem CIDs + SMILES + properties
2. **Download** 3D SDF files from PubChem
3. **Convert** SDF → XYZ via Open Babel (with 3D generation + minimization)
4. **Write** Gaussian `.com` files (opt + freq via Link1)
5. **Generate** SLURM `.sh` scripts for cluster submission

Each step writes a log CSV so you can **resume safely** without redoing completed work.

### Requirements
- Python ≥ 3.10, pandas, requests
- Open Babel (`conda install -c conda-forge openbabel`)
- The `pipeline/` package (included in this repo)

---
## 0 · Configuration

Edit the parameters below to match your **cluster**, **level of theory**, and **directory preferences**.

In [ ]:
import os, sys

# Make sure the pipeline package is importable
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

# ─── Output directories ───────────────────────────────────────────────
MOLECULE_CSV = "molecules_pubchem_smiles.csv"   # Step 1 output
DIAG_CSV     = "molecules_pubchem_diagnostics.csv"
SDF_DIR      = "pubchem_sdf"
XYZ_DIR      = "pubchem_xyz"
COM_DIR      = "gaussian_inputs"
SLURM_DIR    = "slurm_scripts"

# ─── Gaussian parameters ──────────────────────────────────────────────
NPROC = 16

ROUTE_OPT  = "# opt=(tight,calcfc) b3lyp/6-311++g(2df,2p) scrf=(iefpcm,solvent=water)"
ROUTE_FREQ = "# freq b3lyp/6-311++g(2df,2p) scrf=(iefpcm,solvent=water) temperature=298 Geom=AllChk Guess=Read"
TITLE_SUFFIX = "PCM 298 K 6-311++G(2df,2p)"

# ─── SLURM parameters (edit for your cluster) ─────────────────────────
SLURM_ACCOUNT = "myaccount"   # <-- your allocation
SLURM_CPUS    = 16
SLURM_MEM     = "32G"
SLURM_TIME    = "24:00:00"

# ─── PubChem settings ─────────────────────────────────────────────────
CACHE_DIR = ".pubchem_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

for d in (SDF_DIR, XYZ_DIR, COM_DIR, SLURM_DIR):
    os.makedirs(d, exist_ok=True)

print("Working directory:", os.getcwd())

---
## 1 · Define Your Molecules

**Edit the lists below** for your project. See `examples/nucleobases_nucleosides.py` for a more detailed example with aliases and fallback queries.

### Quick guide
- `molecules`: list of human-readable labels
- `alias`: maps a label → the PubChem query string (needed when the common name differs from what PubChem expects)
- `fallback_queries`: if the primary alias fails, these alternatives are tried in order
- `expected_formulas`: optional — helps the scoring heuristic pick the right PubChem record

In [ ]:
# ── Molecule list ──────────────────────────────────────────────────────
# Replace these with your own molecules.

molecules = [
    "Water",
    "Glycine",
    "Adenine",
]

# ── Aliases (optional) ─────────────────────────────────────────────────
# Map label → PubChem name if the label isn't directly resolvable.

alias = {
    # "My Label": "pubchem-resolvable-name",
}

# ── Fallback queries (optional) ───────────────────────────────────────
# If the primary query fails, try these alternatives in order.

fallback_queries = {
    # "My Label": ["alt name 1", "alt name 2"],
}

# ── Expected formulas (optional) ──────────────────────────────────────
# Helps the scoring heuristic pick the right PubChem record.

expected_formulas = {
    # "Adenine": "C5H5N5",
}

---
## 2 · Step 1 — Resolve Molecules via PubChem

Queries PubChem for each molecule, scores candidate records, and writes a CSV table with CIDs, SMILES, and formulas.

In [ ]:
from pipeline.pubchem import build_molecule_table

df, diag = build_molecule_table(
    molecules,
    alias=alias,
    fallback_queries=fallback_queries,
    expected_formulas=expected_formulas,
    cache_dir=CACHE_DIR,
)

display(df)

df.to_csv(MOLECULE_CSV, index=False)
diag.to_csv(DIAG_CSV, index=False)
print(f"Wrote: {MOLECULE_CSV}")
print(f"Wrote: {DIAG_CSV}")

# Check for failures
failed = df[df["cid"].isna()][["name", "pubchem_query", "status", "warnings"]]
if len(failed):
    print("\n⚠ Molecules without CIDs (review these):")
    display(failed)

---
## 3 · Step 2 — Download PubChem SDFs

In [ ]:
from pipeline.pubchem import download_sdfs

sdf_log = download_sdfs(
    input_csv=MOLECULE_CSV,
    sdf_dir=SDF_DIR,
    log_csv="sdf_download_log.csv",
)
display(sdf_log.head())

---
## 4 · Step 3 — Convert SDF → XYZ (Open Babel)

In [ ]:
from pipeline.geometry import convert_sdfs_to_xyz

xyz_log = convert_sdfs_to_xyz(
    download_log_csv="sdf_download_log.csv",
    xyz_dir=XYZ_DIR,
    log_csv="xyz_convert_log.csv",
)
display(xyz_log.head())

---
## 5 · Step 4 — Write Gaussian `.com` Files

Uses the Link1 pattern: optimization writes a checkpoint file, then the frequency job reads it via `Geom=AllChk Guess=Read`.

In [ ]:
from pipeline.gaussian import write_gaussian_coms

com_log = write_gaussian_coms(
    xyz_log_csv="xyz_convert_log.csv",
    outdir=COM_DIR,
    log_csv="com_write_log.csv",
    route_opt=ROUTE_OPT,
    route_freq=ROUTE_FREQ,
    title_suffix=TITLE_SUFFIX,
    nproc=NPROC,
)
display(com_log.head())

---
## 6 · Step 5 — Generate SLURM Scripts

In [ ]:
from pipeline.slurm import write_slurm_scripts

slurm_log = write_slurm_scripts(
    com_dir=COM_DIR,
    slurm_dir=SLURM_DIR,
    log_csv="slurm_write_log.csv",
    account=SLURM_ACCOUNT,
    cpus=SLURM_CPUS,
    mem=SLURM_MEM,
    time=SLURM_TIME,
)
display(slurm_log.head())
print(f"\nDone! Created {len(slurm_log)} SLURM scripts in: {SLURM_DIR}/")

---
## 7 · Next Steps

1. **Transfer** the `gaussian_inputs/` and `slurm_scripts/` directories to your HPC cluster.
2. **Submit** jobs:  
   ```bash
   for f in slurm_scripts/*.sh; do sbatch "$f"; done
   ```
3. **Check convergence** in the `.log` files after jobs complete.
4. **Parse thermochemistry** from the frequency output (ΔG, ΔH, ZPE).

> **Tip:** If a geometry optimization doesn't converge, check the starting structure in the `.xyz` file. You may need to manually adjust the geometry or try a different starting conformation.